In [ ]:
✅ Install (LangChain 1.2+)
pip install -U langchain langchain-community langchain-openai pypdf
📄 Basic PyPDFLoader Usage
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("sample.pdf")

docs = loader.load()

print(len(docs))        # number of pages
print(docs[0].page_content)  # text of first page
print(docs[0].metadata)      # metadata (page number, source, etc.)
📌 Output Structure

Each item in docs is a Document object:

Document(
    page_content="text here...",
    metadata={"source": "sample.pdf", "page": 0}
)
📑 Load and Split Automatically
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("sample.pdf")

docs = loader.load_and_split()

print(len(docs))
✂️ Manual Splitting (Recommended for RAG)
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyPDFLoader("sample.pdf")
docs = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_documents(docs)

print(len(chunks))
🚀 Full RAG Example (PDF → Vector DB → Query)
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. Load PDF
loader = PyPDFLoader("sample.pdf")
docs = loader.load()

# 2. Split
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
chunks = splitter.split_documents(docs)

# 3. Create embeddings
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 4. Store in FAISS
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever()

# 5. Create RAG chain
prompt = ChatPromptTemplate.from_template("""
Answer the question using only the context below:

{context}

Question: {question}
""")

llm = ChatOpenAI(model="gpt-4o-mini")

rag_chain = (
    {"context": retriever, "question": lambda x: x}
    | prompt
    | llm
    | StrOutputParser()
)

# 6. Ask question
response = rag_chain.invoke("What is this document about?")
print(response)
🧠 Async Version
docs = await loader.aload()
⚠️ Common Errors
❌ ModuleNotFoundError: No module named 'pypdf'

Fix:

pip install pypdf
❌ Old import (0.x style)

Do NOT use:

from langchain.document_loaders import PyPDFLoader  # ❌ old

Use:

from langchain_community.document_loaders import PyPDFLoader  # ✅
🔥 Best Practice for Production

✔ Use manual splitting
✔ Store embeddings persistently (FAISS.save_local())
✔ Add metadata filters in retriever
✔ Use k=3–5 for retrieval